# UC-PnC Helpdesk PoC: Llama 3.2 3B (4-bit) answer server

Runs the study's **base** model (`Llama-3.2-3B-Instruct-Q4_K_M`, **not fine-tuned**) with llama.cpp
on Colab's free GPU and opens a temporary public link to it for the PoC.

**Steps**
1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. The first run builds llama.cpp (about 5–10 minutes). Re-runs in the same session skip it.
3. The last cell prints three lines. Copy them into `poc_thesis/.env` (not into any chat), then restart the PoC server.
4. Keep this tab open. When the Colab session ends, the link dies: run all again and copy the **new** lines.

Before sharing or saving this notebook anywhere: **Edit → Clear all outputs** (the output holds the key).

In [ ]:
%%bash
# 1. Build llama.cpp's server with CUDA, for the T4 only (faster build)
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || { echo "No GPU: Runtime > Change runtime type > T4 GPU"; exit 1; }
if [ ! -x /content/llama.cpp/build/bin/llama-server ]; then
  git clone --depth 1 -q https://github.com/ggml-org/llama.cpp /content/llama.cpp
  cd /content/llama.cpp
  cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /content/build.log 2>&1
  cmake --build build --target llama-server -j 4 >> /content/build.log 2>&1 || { tail -30 /content/build.log; exit 1; }
fi
echo "llama-server ready: $(/content/llama.cpp/build/bin/llama-server --version 2>&1 | head -1)"

In [ ]:
# 2. Download the model file (about 2 GB, public, no token needed)
from huggingface_hub import hf_hub_download
MODEL = hf_hub_download("hugging-quants/Llama-3.2-3B-Instruct-Q4_K_M-GGUF",
                        "llama-3.2-3b-instruct-q4_k_m.gguf", local_dir="/content/models")
print(MODEL)

In [ ]:
# 3. Start the server on this machine only, protected by a fresh random key
import secrets, subprocess, time, urllib.request
try: server.terminate()
except NameError: pass
subprocess.run(["pkill", "-f", "llama-server"])   # a server left over from an earlier run
time.sleep(1)
# Port 8181, not 8080: Colab uses 8080 for itself.
KEY = secrets.token_urlsafe(24)
server = subprocess.Popen(
    ["/content/llama.cpp/build/bin/llama-server", "-m", MODEL,
     "--alias", "Llama-3.2-3B-Instruct-Q4_K_M",   # must match LLAMA_MODEL in server/config.py
     "--host", "127.0.0.1", "--port", "8181",
     "-ngl", "99",                               # whole model on the GPU
     "-c", "4096", "-np", "1",                   # one request at a time, 4k context
     "--api-key", KEY],
    stdout=open("/content/llama-server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if urllib.request.urlopen("http://127.0.0.1:8181/health", timeout=2).status == 200: break
    except Exception: pass
    if server.poll() is not None: break
    time.sleep(1)
if server.poll() is not None:
    print(open("/content/llama-server.log").read()[-3000:]); raise SystemExit("server stopped")
print("server is up")

In [ ]:
# 4. Open a temporary public link (Cloudflare quick tunnel, no account needed)
import os, re, subprocess, time
if not os.path.exists("/content/cloudflared"):
    subprocess.run("wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared", shell=True, check=True)
try: tunnel.terminate()
except NameError: pass
tlog = "/content/tunnel.log"
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8181"],
                          stdout=open(tlog, "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(tlog).read())
    if m: URL = m.group(0); break
    time.sleep(1)
if not URL: print(open(tlog).read()[-2000:]); raise SystemExit("no tunnel URL")
time.sleep(5)   # give the new link a moment to go live
print("tunnel:", URL)

In [ ]:
# 5. Test through the public link, then print the lines for poc_thesis/.env
import json, time, urllib.request
req = urllib.request.Request(URL + "/v1/chat/completions", method="POST",
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + KEY},
    data=json.dumps({"model": "Llama-3.2-3B-Instruct-Q4_K_M", "temperature": 0, "max_tokens": 20,
                     "messages": [{"role": "user", "content": "Say: ready"}]}).encode())
for attempt in range(6):
    try:
        t0 = time.time()
        r = json.load(urllib.request.urlopen(req, timeout=60))
        print(f"test reply: {r['choices'][0]['message']['content']!r}  ({time.time()-t0:.1f} s)")
        break
    except Exception as e:
        print("waiting for the link…", type(e).__name__); time.sleep(5)
print()
print("Copy these 3 lines into poc_thesis/.env, then restart the PoC server:")
print()
print("POC_GENERATOR=llama")
print("LLAMA_BASE_URL=" + URL)
print("LLAMA_API_KEY=" + KEY)